The same lab as a web page, with the folded answers: https://ml.wxing.me/materials/lesson6/lab.html

# Lab 6: Grow a Tree, Find What Goes Wrong, Fix It
## MPS311/439 Machine Learning
**Lesson 6 Lab Session | Core duration: 50 minutes**

## What this lab is for

**This lab has no blanks.** All the code is given; you decide what to change and what to look at. Your job is to run the code, notice what goes wrong with a decision tree, and try a fix.

Each Part follows the same six steps, the same steps the assignments are marked on:

1. **Observe**: run the cell and write one sentence about what you see. Quote a number from the output.
2. **Idea**: guess why it happens.
3. **Action**: change the one value marked `← change`, or run the next cell.
4. **Test**: run again with the same folds `kf`, and compare with the number before.
5. **Explain**: say why the result changed (or did not).
6. **Limits**: say where your conclusion stops.

**You write only two short answers per Part**: Observe (fill in the numbers from your own output) and one sentence that Explains and gives the Limits. For Limits, a sentence like "this holds for ..., but may not for ..." is enough. Idea, Action and Test you do as you go: make your guess, change the value, run, compare. There is no single right answer. A good answer quotes a number from your own output and changes one thing at a time. Each Part also has a **Think about** question to discuss with your neighbour (no writing), **one possible answer** (folded) and a **Challenge** for when you finish early. The **Advanced Study** at the end is for after class.

## How to work

1. This notebook already has all the code. Save a copy first (**File → Save a copy in Drive**). Run the cells from top to bottom: later cells use names made by earlier ones.
2. Write your six-step answers in a text cell (**+ Text**) under the code.
3. A few **Run, and you should see** lines, where a mistake would be hard to notice, tell you what your output should show. Everywhere else, read the numbers from your own output; the folded answers have all of them if you want to compare afterwards.
4. The **Try with AI** boxes are folded: open one when you finish early, or after class. They do not count in the 50 minutes.

### Using AI

The **Try with AI** boxes let an AI assistant write the Python you do not know yet (a plot, a picture of the boundary), or explain an idea in other words, while you stay in charge. Every box has the same four steps:

1. **You want:** what you want, in machine-learning words.
2. **Tell the AI:** the names of the variables it should use.
3. **Ask:** the request.
4. **Check:** compare the AI's result with a number on this page, or with your own output. If they do not match, the AI or your question is wrong: find out which.

## Part 0: Set up and grow a first tree (6 minutes)

### 0.1 The setup cell

Run this cell without reading it closely. It is the same at the start of every lab.

In [ ]:
import numpy as np                                        # numbers and arrays
import pandas as pd                                       # tables
import matplotlib.pyplot as plt                           # plots
from sklearn.datasets import load_iris                    # today's dataset
from sklearn.model_selection import train_test_split, KFold, cross_val_score, cross_val_predict   # splitting and cross-validation
from sklearn.preprocessing import StandardScaler          # puts every column on one scale
from sklearn.tree import DecisionTreeClassifier, plot_tree   # today's model, and a picture of it
from sklearn.linear_model import LogisticRegression       # a straight-line classifier, for comparison

### 0.2 The data block

You do not change anything in this block today. **The data.** 150 iris flowers of three species (0 setosa, 1 versicolor, 2 virginica). Each flower has four measurements in cm: sepal length, sepal width, petal length, petal width.

In [ ]:
data = load_iris()                                        # ← change: which dataset (not today)
X = data.data                                             # the measurements (features)
y = data.target                                           # ← change: the labels (not today; here the species 0, 1, 2)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)     # 3:1 split, the same mix of labels in both parts
scaler = StandardScaler()                                 # one scale for every column
X_train_s = scaler.fit_transform(X_train)                 # learn the scale from the training data only
X_test_s = scaler.transform(X_test)                       # apply the same scale to the test data
kf = KFold(5, shuffle=True, random_state=0)               # the same 5 folds every time
print(X_train.shape, np.bincount(y_train))                # training rows and columns; how many of each label

**Run, and you should see:** `(112, 4) [37 37 38]`.

### 0.3 A first tree

In [ ]:
tree = DecisionTreeClassifier(max_depth=2, random_state=0)   # a tree that asks at most 2 questions
print(cross_val_score(tree, X_train, y_train, cv=kf).mean())  # 5-fold cross-validated accuracy

**Run, and you should see:** about `0.955`. A tree does not need the standardised `X_train_s`: each question compares one measurement with one number.

> **Challenge (if you finish early)**
>
> The tree scores each cut by Gini. Add `criterion="entropy"` inside `DecisionTreeClassifier(...)` and run again. Does the accuracy change? Draw both trees with `plot_tree` (Part 3.1 shows how): is the first cut the same?

## Part 1: The tree cuts too deep (15 minutes)

The lecture grew its trees on the Titanic passengers; here you see the same problems on a different dataset. With all four measurements the species are easy to tell apart. To see what goes wrong with deep trees, use only the two **sepal** measurements, where versicolor and virginica overlap.

### 1.1 Grow the tree, then change its depth

In [ ]:
X_sepal = X_train[:, :2]                                  # only columns 0 and 1: sepal length and width
tree = DecisionTreeClassifier(max_depth=None, random_state=0)   # ← change: None means no depth limit
tree.fit(X_sepal, y_train)                                # grow the tree on all 112 training flowers
print(tree.get_n_leaves(), tree.score(X_sepal, y_train))  # number of leaves, training accuracy
print(cross_val_score(tree, X_sepal, y_train, cv=kf).mean())    # 5-fold cross-validated accuracy

**Run, and you should see:** `37` leaves, training accuracy about `0.946`, cross-validated accuracy about `0.696`. (Training accuracy is not 1.0 because some flowers have exactly the same sepal length and width but different species: no cut can separate them.)

**Observe** (fill in): `Leaves: ___   Training: ___   Cross-validated: ___   Gap: ___`. Then change `max_depth=None` to `3`, run again, and look at the gap. Guess, without writing, why the deep tree does worse on flowers it did not see.

*Your answer:* 

### 1.2 Draw the two curves

In [ ]:
depths = [1, 2, 3, 4, 5, 6, 8, 10]                        # ← change: the depths to try
train_list = []                                           # training accuracy, one per depth
cv_list = []                                              # cross-validated accuracy, one per depth
for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0)
    tree.fit(X_sepal, y_train)                            # fit on all training flowers
    train_list.append(tree.score(X_sepal, y_train))       # accuracy on the flowers it was fitted on
    cv_list.append(cross_val_score(tree, X_sepal, y_train, cv=kf).mean())   # accuracy on flowers it did not see
    print(depth, round(train_list[-1], 3), round(cv_list[-1], 3))
plt.plot(depths, train_list, "o-", label="training")      # one line per kind of accuracy
plt.plot(depths, cv_list, "s-", label="5-fold cross-validation")
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.legend()
plt.show()

Choose the depth with the highest cross-validated accuracy, reading the printed numbers; when two depths tie, take the shallower one, the simpler tree. Note it (just the number). Is it at an end of the range in `depths`? If it were, add more depths beyond that end and run again. (In Assignment 2 the validation set is given and the data are large, so you will not use `cross_val_score` there; use the validation-set loop of Part 1.3 after class.)

> **Try with AI (if you finish early, or after class)**
>
> 1. **You want:** the same figure, but with the accuracy on one validation set (the after-class Part 1.3) as a third line.
> 2. **Tell the AI:** your code from 1.2 and from 1.3 after class, and the names `X_fit`, `y_fit`, `X_val`, `y_val`.
> 3. **Ask:** "Add a third line to my plot: for each depth, fit the tree on `X_fit`, `y_fit` and score it on `X_val`, `y_val`. Keep my style: one step per line with a comment."
> 4. **Check:** the new line peaks at depth 4 (0.714) and is more jagged than the cross-validated line. Assignment 2's optional figure is this plot with the training and validation lines.

**Explain and Limits** (one sentence): why does the deep tree score so much better on the flowers it was fitted on than under cross-validation, and how far does your choice of depth hold?

*Your answer:* 

**Think about** (with your neighbour, no writing): a tree scores 100% on its training data. Is that good news?

> **Try with AI (if you finish early, or after class)**
>
> 1. **You want:** a picture of the boxes the tree cuts in the sepal plane, for a deep tree and a shallow one.
> 2. **Tell the AI:** `X_sepal`, `y_train`, and two trees: `DecisionTreeClassifier(max_depth=None, random_state=0)` and `DecisionTreeClassifier(max_depth=3, random_state=0)`.
> 3. **Ask:** "Write simple Python that fits each tree on `X_sepal`, `y_train` and draws its decision regions with `sklearn.inspection.DecisionBoundaryDisplay`, with the training flowers on top coloured by species, side by side."
> 4. **Check:** the deep tree's picture has many small boxes, some around one or two flowers; the depth-3 tree has a few large boxes. If the boxes are not straight, horizontal and vertical lines, the picture is wrong.

> **Challenge (if you finish early)**
>
> `max_depth` is one way to stop a tree. Another is `min_samples_leaf`: every leaf must keep at least that many flowers. In 1.1, replace `max_depth=None` with `min_samples_leaf=5`, then try `10`, `20` and `30`. Which value gives the best cross-validated accuracy, and why does `30` suddenly fall so low? (Count how many flowers of each species there are.)

## Part 2: Can you trust feature importance? (14 minutes)

Now use all four measurements again.

### 2.1 Which measurements does the tree use?

In [ ]:
names = ["sepal length", "sepal width", "petal length", "petal width"]   # the four columns, in order
tree = DecisionTreeClassifier(max_depth=3, random_state=0)   # a tree that asks at most 3 questions
print(cross_val_score(tree, X_train, y_train, cv=kf).mean())  # 5-fold cross-validated accuracy
tree.fit(X_train, y_train)                                # fit on all training flowers
print(pd.Series(tree.feature_importances_, index=names).round(3))   # importance: each column's share of the Gini drop
print(np.corrcoef(X_train[:, 2], X_train[:, 3])[0, 1])    # correlation of petal length and petal width

Recall: Gini measures how mixed a box is (0 means one species only). Every cut lowers it; a column's importance is its share of all that lowering, and the shares add up to 1.

**Observe** (fill in): `Relies on (one column): ___   Petal length's importance: ___   Correlation of the two petal measurements: ___`. Then guess, without writing: if you took petal **width** away, would the importance of petal length go up or down? Use the correlation.

*Your answer:* 

### 2.2 Take one measurement away

In [ ]:
keep = [0, 1, 2]                                          # ← change: the columns to keep (here: all but petal width, column 3)
X_less = X_train[:, keep]                                 # the training flowers with only these columns
tree_less = DecisionTreeClassifier(max_depth=3, random_state=0)   # the same kind of tree as in 2.1
print(cross_val_score(tree_less, X_less, y_train, cv=kf).mean())   # the same 5 folds as before
tree_less.fit(X_less, y_train)                            # fit without the removed column
kept_names = np.array(names)[keep]                        # picks out the names of the columns in keep
print(pd.Series(tree_less.feature_importances_, index=kept_names).round(3))

Run it: this tree never sees petal width. This is the same move as removing class in Section 8 of the Notes. Compare the accuracy and the importance of petal length with 2.1.

**Explain and Limits** (one sentence): how did petal length's importance change, and why? What does an importance close to 0 tell you, and what does it not tell you?

*Your answer:* 

**Think about** (with your neighbour, no writing): you are a botanist with a ruler and one minute per flower. Which single measurement would you take, and what in today's output tells you so?

> **Try with AI (if you finish early, or after class)**
>
> 1. **You want:** an explanation, in other words, of why two almost identical features share a tree's importance.
> 2. **Tell the AI:** your own numbers: petal width 0.968 and petal length 0.032 in the full tree; correlation 0.964; without petal width, petal length 0.986 and accuracy 0.892.
> 3. **Ask:** "Explain to a student, in five sentences and without code, why a decision tree gives almost all the importance to one of two highly correlated features. Use my numbers."
> 4. **Check:** the explanation must say that each cut uses only one feature, so the first one chosen takes the credit, and that a small importance does not mean the feature is useless. If it says petal length is useless, it contradicts your 0.892.

> **Challenge (if you finish early)**
>
> Set `keep = [0, 1]`: only the two sepal measurements, which had importance 0 in 2.1. What is the accuracy now, and which of the two takes most of the credit? Is "the tree never used the sepals" evidence that sepals say nothing about the species?

## Part 3: How sure is the tree? (13 minutes)

### 3.1 Read the tree

In [ ]:
tree = DecisionTreeClassifier(max_depth=2, random_state=0)   # the tree of Part 0
tree.fit(X_train, y_train)                                # fit on all 112 training flowers
plt.figure(figsize=(14, 7))                               # a large figure, so the boxes are readable
plot_tree(tree, feature_names=names, class_names=["setosa", "versicolor", "virginica"], filled=True, fontsize=12)   # draw it
plt.show()

**Run, and you should see:** a root that asks about petal width, and three leaves. Each box shows its question (not in leaves), the Gini of that box, `samples` (how many training flowers reach it) and `value` (how many of each species). One leaf is all setosa (`value = [37, 0, 0]`). Find the leaf whose `value = [0, 36, 3]`: it mixes two species, so 36 of its 39 training flowers are versicolor, and the tree gives any flower landing there a probability of 36/39 = 0.923 for versicolor.

### 3.2 The tree's probabilities

A leaf's shares are the tree's probabilities. `cross_val_predict` gives every training flower the probabilities from a tree that did not see that flower. We call a flower "not sure" when its highest probability is below 0.95; 0.95 is our own choice.

In [ ]:
tree = DecisionTreeClassifier(max_depth=2, random_state=0)   # ← change: try None
proba = cross_val_predict(tree, X_train, y_train, cv=kf, method="predict_proba")   # 3 probabilities per flower
pred = proba.argmax(axis=1)                               # the species with the highest probability
top = proba.max(axis=1)                                   # each flower's highest probability
print("not sure (top probability below 0.95):", (top < 0.95).sum())   # count the flowers below 0.95
print("wrong:", (pred != y_train).sum())                 # count the flowers predicted wrongly

Run it at depth 2, then set `max_depth=None` and run again. **If every flower now looks sure, you did not make a mistake**: that is what this part is about.

**Observe** (fill in): `Depth 2: not sure ___, wrong ___.   No limit: not sure ___, wrong ___.` If the answer surprises you, check that you changed only `max_depth`.

*Your answer:* 

**Explain and Limits** (one sentence): can you trust the probabilities of a deep tree?

*Your answer:* 

### 3.3 The test set, once

Make your last choice: which depth would you use for all four measurements? Look back at Part 0 (depth 2 scored 0.955) and at 3.2. Put it in the cell, then read the test set once.

In [ ]:
tree = DecisionTreeClassifier(max_depth=2, random_state=0)   # ← change: the depth you choose
tree.fit(X_train, y_train)                                # fit on all 112 training flowers
print(tree.score(X_test, y_test))                         # the 38 test flowers, read once

**Run, and you should see:** about `0.947` if you chose depth 2; any depth from 2 to 4 is a choice you can defend from Parts 0 and 3. Whatever you chose, do not change it after seeing this number: that would be choosing on the test set.


> **Challenge (if you finish early)**
>
> In 3.2, use `DecisionTreeClassifier(min_samples_leaf=10, random_state=0)`. How many flowers are not sure now, and how many are wrong? Then replace the tree with `LogisticRegression(max_iter=1000)`, and in the `cross_val_predict` line use `X_train_s` instead of `X_train` (logistic regression needs the standardised data). Which of the three has the most flowers "not sure" (the most cautious), and which is wrong least often?

# After class: one small investigation, reflection, and Assignment 2

This part goes with Assignment 2, Section 2: there you choose `max_depth` on one validation set. Run it after Part 1 (it uses `X_sepal`).

## Part 1.3: Choose the depth with one validation set

Cross-validation fits 5 trees per depth. A single validation set fits only one: hold out 28 of the 112 flowers, fit on the other 84, and score each depth on the 28.

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(
    X_sepal, y_train, test_size=0.25, random_state=0, stratify=y_train)   # one validation set of 28 flowers
for depth in [1, 2, 3, 4, 5, 6, 7, 8]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0)
    tree.fit(X_fit, y_fit)                                # fit on the 84 fitting flowers
    print(depth, round(tree.score(X_val, y_val), 3))      # accuracy on the 28 validation flowers

Find the depth with the best validation accuracy and compare it with your choice in 1.2. We also ran the same cell with two other validation sets (`random_state=1` and `2`): they pick depth `5` and depth `3`.

**Explain and Limits** (one sentence): why do the three validation sets pick different depths? (One flower is 1/28 = 0.036 of a validation accuracy: how many flowers does it take to change which depth wins?) Then: Assignment 2 has 10,000 validation images, where one image is 0.0001 of the accuracy. Would one validation set be good enough there?

*Your answer:* 

## A small investigation on the Titanic passengers

The Note grew its trees on the Titanic passengers. Load them with the same data block (only the two `← change` lines and two lines of cleaning are new):

In [ ]:
ship = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")   # 891 passengers
ship = ship[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare"]].dropna()   # 6 features and the label; keep the 714 with a known age
ship["sex"] = (ship["sex"] == "female").astype(int)       # 1 = female, 0 = male
X = ship.drop(columns="survived").values                  # ← change: the 6 features
y = ship["survived"].values                               # ← change: 1 = survived, 0 = died
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
kf = KFold(5, shuffle=True, random_state=0)
print(X_train.shape, np.bincount(y_train))

**Run, and you should see:** `(535, 6) [318 217]`.

In [ ]:
tree = DecisionTreeClassifier(max_depth=4, random_state=0)   # the depth chosen in the Note
print(cross_val_score(tree, X_train, y_train, cv=kf).mean())
line = LogisticRegression(max_iter=1000)                  # a straight line, on the standardised features
print(cross_val_score(line, X_train_s, y_train, cv=kf).mean())

Write half a page, in the six steps: which method would you choose for these passengers, why, and when would you change your mind? Use at least one more run of your own as evidence (another depth, another feature removed, `plot_tree`). This is the kind of answer the assignments reward at the "good" and "excellent" levels: an idea that comes from your own numbers, a fair test, and a conclusion that says where it stops.

## Reflection

One or two sentences each, after class.

1. What does a decision tree's `.fit()` actually do? Describe it in your own words: what does it try, and what does it keep?
2. A tree scores 100% on its training data and 75% under cross-validation. Is it a good model? What would you change?
3. A feature has importance 0.0. Does that mean it is useless for predicting the label? How would you check?
4. When would you choose a decision tree over logistic regression, and when not?

## How today's lab helps with Assignment 2

| What you did today | Where it appears in Assignment 2 |
|---|---|
| Part 1.2: try several `max_depth` values; the best must not sit at an end | Section 2: "Choose both using the validation set only ... The chosen value must not lie at either end of your search range" |
| Part 1.3 (after class): choose the depth on one validation set | Section 2: with 50,000 images the assignment uses one validation set. Use the loop of 1.3 with the assignment's `X_train`, `y_train`, `X_val`, `y_val`; with 10,000 validation images one image changes the accuracy by 0.0001, so the pick does not depend on luck |
| Part 1.2: training and cross-validated accuracy against depth | Section 2, Going further: the figure of training and validation accuracy against depth |
| Part 3.3: read the test set once | Section 4: `test` is used once, at the end |
| The six steps of every Part | The marking standard of every assignment |